# SupplySight AI

# Enterprise Procurement Decision Intelligence Platform

## Notebook 03B

# External Data Cleaning & Standardization

---

## Objective

The objective of this notebook is to validate, clean and standardize all external datasets used in the procurement intelligence platform.

Unlike the transactional ERP dataset, these datasets act as reference and enrichment data.

The notebook focuses on:

- Data Quality Assessment
- Column Standardization
- Data Type Conversion
- Duplicate Detection
- Missing Value Validation
- Dataset Standardization
- Processed Dataset Generation

The cleaned datasets generated in this notebook will later be integrated with procurement transactions during feature engineering.

# Import Libraries

In [46]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from pathlib import Path

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


# Project Paths

In [47]:
PROJECT_PATH = Path(r"C:\Users\chavi\Desktop\SupplySight AI – Enterprise Procurement Decision Intelligence Platform")

RAW = PROJECT_PATH / "Datasets" / "Raw Datasets"

EXTERNAL = PROJECT_PATH / "Datasets" / "External Datasets"

PROCESSED = PROJECT_PATH / "Datasets" / "Processed Datasets"

PROCESSED.mkdir(exist_ok=True)

# Load External Datasets

In [3]:
# ============================================================
# Load External Datasets
# ============================================================

# Commodity Prices
commodity = pd.read_excel(
    RAW / "CMO-Historical-Data-Monthly.xlsx",
    sheet_name="Monthly Prices",
    header=4
)

# Fuel Prices
fuel = pd.read_csv(
    EXTERNAL / "fuel_prices.csv"
)

# Exchange Rates
exchange_inr = pd.read_csv(
    EXTERNAL / "exchange_inr.csv"
)

exchange_eur = pd.read_csv(
    EXTERNAL / "exchange_eur.csv"
)

# Enterprise Country Dimension
dim_country = pd.read_csv(
    PROCESSED / "dim_country.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

# Enterprise Weather Dimension
dim_weather = pd.read_csv(
    PROCESSED / "dim_weather.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

# Enterprise Holiday Dimension
dim_holiday = pd.read_csv(
    PROCESSED / "dim_holiday.csv",
    dtype={"iso2": str},
    keep_default_na=False
)

print("=" * 70)
print("ENTERPRISE EXTERNAL DATASETS LOADED")
print("=" * 70)

ENTERPRISE EXTERNAL DATASETS LOADED


# Dataset Inventory

In [4]:
inventory = pd.DataFrame({

    "Dataset":[
        "Commodity",
        "Fuel",
        "Exchange USD-INR",
        "Exchange USD-EUR",
        "Country Dimension",
        "Weather Dimension",
        "Holiday Dimension"
    ],

    "Rows":[
        commodity.shape[0],
        fuel.shape[0],
        exchange_inr.shape[0],
        exchange_eur.shape[0],
        dim_country.shape[0],
        dim_weather.shape[0],
        dim_holiday.shape[0]
    ],

    "Columns":[
        commodity.shape[1],
        fuel.shape[1],
        exchange_inr.shape[1],
        exchange_eur.shape[1],
        dim_country.shape[1],
        dim_weather.shape[1],
        dim_holiday.shape[1]
    ]

})

inventory

,Dataset,Rows,Columns
0,Commodity,798,72
1,Fuel,2299,5
2,Exchange USD-INR,1043,2
3,Exchange USD-EUR,1043,2
4,Country Dimension,164,14
5,Weather Dimension,239604,9
6,Holiday Dimension,239604,8


### Dataset Overview

The external data sources consist of reference and enrichment datasets that complement the enterprise procurement transactions.

| Dataset | Records | Columns | Purpose |
|---------|--------:|--------:|---------|
| Fuel Prices | 5,000 | 5 | Fuel price trends for logistics and transportation cost analysis. |
| Holiday | 220 | 5 | Public holiday calendar for demand forecasting and operational planning. |
| Commodity | 798 | 72 | Historical commodity prices used for procurement cost analysis. |
| Exchange USD-INR | 1,305 | 2 | Daily USD to INR exchange rates for currency normalization. |
| Exchange USD-EUR | 1,305 | 2 | Daily USD to EUR exchange rates for international procurement analysis. |
| Weather | 328,736 | 7 | Historical weather observations for external demand and supply chain impact analysis. |

### Observations

- A total of **6 external datasets** have been successfully loaded.
- The datasets vary considerably in size, ranging from **220 holiday records** to more than **328,000 weather observations**.
- Each dataset represents a different external business factor that can influence procurement decisions.
- The commodity dataset contains the largest number of features, indicating that additional preprocessing and restructuring may be required.
- Exchange rate datasets have a simple structure and will primarily require date standardization before integration.
- Weather data is the largest dataset and may require filtering or aggregation during feature engineering to improve computational efficiency.

### Next Steps

Each external dataset will undergo the following quality assurance process:

- Data quality assessment
- Column name standardization
- Data type validation and conversion
- Duplicate detection
- Missing value validation
- Dataset-specific standardization
- Export of cleaned datasets for feature engineering

### Outcome

The external datasets have been successfully loaded and inventoried. They are ready for detailed data quality assessment and standardization before being integrated with the enterprise procurement dataset.

# Helper Functions

In [5]:
def standardize_columns(df):

    df.columns = (
        df.columns
          .str.lower()
          .str.strip()
          .str.replace(" ", "_", regex=False)
          .str.replace("-", "_", regex=False)
          .str.replace("/", "_", regex=False)
    )

    return df

In [6]:
def data_quality_report(df, dataset_name):

    report = pd.DataFrame({
        "Data Type": df.dtypes,
        "Missing Values": df.isnull().sum(),
        "Missing %": round(df.isnull().mean()*100,2),
        "Unique Values": df.nunique()
    })

    print("="*80)
    print(dataset_name.upper())
    print("="*80)

    display(report)

    print("Duplicate Rows :", df.duplicated().sum())

### Functions Implemented

#### 1. `standardize_columns()`

This function standardizes column names by applying a consistent naming convention across all datasets.

The following transformations are performed:

- Convert column names to lowercase.
- Remove leading and trailing whitespace.
- Replace spaces (` `) with underscores (`_`).
- Replace hyphens (`-`) with underscores (`_`).
- Replace forward slashes (`/`) with underscores (`_`).

This ensures that all datasets follow the **snake_case** naming convention, making them easier to reference in Python and SQL.

---

#### 2. `data_quality_report()`

This function generates a comprehensive data quality summary for any dataset.

The report includes:

- Data type of each column.
- Number of missing values.
- Percentage of missing values.
- Number of unique values.
- Total duplicate records.

Using a single reusable function ensures that every external dataset is assessed using the same validation criteria, promoting consistency throughout the preprocessing pipeline.

### Benefits

Implementing reusable helper functions provides several advantages:

- Eliminates repetitive code across multiple datasets.
- Ensures consistent preprocessing and reporting.
- Simplifies notebook maintenance and future enhancements.
- Supports scalable processing as additional external datasets are introduced.
- Improves readability and reproducibility of the data cleaning workflow.

### Outcome

The preprocessing utilities have been successfully defined and will be reused throughout this notebook to standardize column names and perform consistent data quality assessments across all external datasets.

# Standardize Column Names

In [7]:
fuel = standardize_columns(fuel)
dim_holiday = standardize_columns(dim_holiday)
commodity = standardize_columns(commodity)
exchange_inr = standardize_columns(exchange_inr)
exchange_eur = standardize_columns(exchange_eur)
dim_country = standardize_columns(dim_country)
dim_weather = standardize_columns(dim_weather)

### Method Applied

The reusable `standardize_columns()` function was applied to each external dataset:

- Fuel Prices
- Holiday
- Commodity Prices
- Exchange Rates (USD-INR)
- Exchange Rates (USD-EUR)
- Weather

The function performs the following transformations:

- Converts all column names to lowercase.
- Removes leading and trailing whitespace.
- Replaces spaces (` `) with underscores (`_`).
- Replaces hyphens (`-`) with underscores (`_`).
- Replaces forward slashes (`/`) with underscores (`_`).

### Benefits

Standardizing column names provides several advantages:

- Establishes a consistent **snake_case** naming convention.
- Simplifies referencing columns in Python and SQL.
- Reduces the likelihood of syntax errors caused by spaces or special characters.
- Improves consistency when joining multiple datasets during feature engineering.

### Outcome

All external datasets now follow a standardized column naming convention, ensuring consistency throughout the procurement intelligence pipeline.

# Data Quality Assessment

In [8]:
datasets = {
    "Commodity": commodity,
    "Fuel": fuel,
    "Exchange USD-INR": exchange_inr,
    "Exchange USD-EUR": exchange_eur,
    "Country Dimension": dim_country,
    "Weather Dimension": dim_weather,
    "Holiday Dimension": dim_holiday
}

for name, df in datasets.items():
    data_quality_report(df, name)

COMMODITY


,Data Type,Missing Values,Missing %,Unique Values
unnamed:_0,str,1,0.13,797
"crude_oil,_average",object,0,0.00,580
"crude_oil,_brent",object,0,0.00,556
"crude_oil,_dubai",object,0,0.00,556
"crude_oil,_wti",object,0,0.00,496
...,...,...,...,...
nickel,object,0,0.00,588
zinc,object,0,0.00,761
gold,object,0,0.00,705
platinum,object,0,0.00,636


Duplicate Rows : 0
FUEL


,Data Type,Missing Values,Missing %,Unique Values
date,str,0,0.0,209
region,str,0,0.0,11
fuel_type,str,0,0.0,1
fuel_price,float64,0,0.0,1191
unit,str,0,0.0,1


Duplicate Rows : 0
EXCHANGE USD-INR


,Data Type,Missing Values,Missing %,Unique Values
date,str,0,0.00,1043
usd_inr_rate,float64,43,4.12,538


... (5 outputs left out)

,Data Type,Missing Values,Missing %,Unique Values
date,str,0,0.0,1461
iso2,str,0,0.0,164
country,str,0,0.0,164
city,str,0,0.0,164
avg_temperature,float64,0,0.0,711
precipitation,float64,0,0.0,925
rainfall,float64,0,0.0,923
snowfall,float64,0,0.0,211
max_wind_speed,float64,0,0.0,584


Duplicate Rows : 0
HOLIDAY DIMENSION


,Data Type,Missing Values,Missing %,Unique Values
date,str,0,0.0,1461
iso2,str,0,0.0,164
country_name,str,0,0.0,164
erp_country,str,0,0.0,164
holiday_name,str,0,0.0,1933
is_holiday,int64,0,0.0,2
is_weekend,int64,0,0.0,2
holiday_source_available,int64,0,0.0,2


Duplicate Rows : 0


## Observations

| Dataset | Key Observations | Planned Action |
|----------|------------------|----------------|
| **Fuel Prices** | No missing values or duplicate records. The `date` column is stored as a string and requires conversion to a datetime format. | Convert the `date` column to `datetime`. |
| **Holiday** | No missing values or duplicate records. The `holiday_date` column is stored as a string. | Convert `holiday_date` to `datetime`. |
| **Commodity Prices** | One missing value (0.13%) exists in the first column (`unnamed:_0`). Most commodity price columns are stored as `object` data types instead of numeric values. No duplicate records were found. | Rename the first column to `date`, remove the incomplete record, convert the `date` column to `datetime`, and convert all commodity price columns to numeric data types. |
| **Exchange USD-INR** | The `usd_inr_rate` column contains **43 missing values (4.12%)**, which correspond to weekends and market holidays. The `date` column is stored as a string. | Convert `date` to `datetime` and fill missing exchange rates using forward-fill (`ffill`). |
| **Exchange USD-EUR** | The `usd_eur_rate` column contains **43 missing values (4.12%)**, which correspond to weekends and market holidays. The `date` column is stored as a string. | Convert `date` to `datetime` and fill missing exchange rates using forward-fill (`ffill`). |
| **Weather** | No missing values or duplicate records. The dataset contains complete daily weather observations for all selected cities. The `date` column is stored as a string. | Convert the `date` column to `datetime`. |

## Summary

The external datasets exhibit a high level of data quality, with no duplicate records and minimal missing values. Most preprocessing tasks involve standardizing date columns, converting data types, and handling expected missing values in the exchange rate datasets caused by non-trading days. The commodity dataset additionally requires renaming its first column and converting commodity price fields from object to numeric data types. After these preprocessing steps, all datasets will be standardized and ready for integration into the enterprise procurement analytics platform during the feature engineering stage.

# Data Type Standardization

In [9]:
# Fuel Prices
fuel["date"] = pd.to_datetime(
    fuel["date"],
    errors="coerce"
)

# Holiday
dim_holiday["date"] = pd.to_datetime(
    dim_holiday["date"],
    errors="coerce"
)

# Exchange Rates (USD-INR)
exchange_inr["date"] = pd.to_datetime(
    exchange_inr["date"],
    errors="coerce"
)

# Exchange Rates (USD-EUR)
exchange_eur["date"] = pd.to_datetime(
    exchange_eur["date"],
    errors="coerce"
)

#Weather
dim_weather["date"] = pd.to_datetime(dim_weather["date"])

### Standardization Performed

| Dataset | Date Column | Action |
|----------|-------------|--------|
| Fuel Prices | `date` | Converted from string to `datetime`. |
| Holiday | `holiday_date` | Converted from string to `datetime`. |
| Exchange USD-INR | `date` | Converted from string to `datetime`. |
| Exchange USD-EUR | `date` | Converted from string to `datetime`. |
| Weather | `date` | Converted from string to `datetime`. |

### Notes

- The `errors="coerce"` parameter is used for datasets containing string-based dates to safely handle any invalid or malformed date values by converting them to `NaT`.
- The weather dataset already contains valid date values, allowing direct conversion to the `datetime` data type.
- After this step, all datasets use standardized date formats, ensuring compatibility with downstream feature engineering and time-series analysis.

# Duplicate Analysis

In [10]:
for name, df in datasets.items():

    print(f"{name}: {df.duplicated().sum()} duplicate rows")

Commodity: 0 duplicate rows
Fuel: 0 duplicate rows
Exchange USD-INR: 0 duplicate rows
Exchange USD-EUR: 0 duplicate rows
Country Dimension: 0 duplicate rows
Weather Dimension: 0 duplicate rows
Holiday Dimension: 0 duplicate rows


### Observations

| Dataset | Duplicate Rows | Observation |
|----------|---------------:|-------------|
| Fuel Prices | 0 | No duplicate records were detected. |
| Holiday | 0 | Dataset contains unique holiday records. |
| Commodity Prices | 0 | No duplicate commodity price records were found. |
| Exchange USD-INR | 0 | Exchange rate observations are unique. |
| Exchange USD-EUR | 0 | Exchange rate observations are unique. |
| Weather | 0 | Daily weather observations are unique for each city and date. |

### Summary

The duplicate analysis confirms that all external datasets are free from duplicate records. As no duplicate entries were identified, no deduplication is required. The datasets maintain record-level uniqueness and are suitable for subsequent data cleaning and feature engineering tasks.

# Missing Value Analysis

In [11]:
for name, df in datasets.items():

    missing = (
        df.isnull()
          .sum()
          .to_frame("Missing Values")
    )

    missing["Missing %"] = (
        missing["Missing Values"] / len(df) * 100
    ).round(2)

    missing = missing[
        missing["Missing Values"] > 0
    ]

    print("="*70)
    print(name)

    display(missing)

Commodity


,Missing Values,Missing %
unnamed:_0,1,0.13


Fuel


,Missing Values,Missing %


Exchange USD-INR


,Missing Values,Missing %
usd_inr_rate,43,4.12


... (4 outputs left out)

Weather Dimension


,Missing Values,Missing %


Holiday Dimension


,Missing Values,Missing %


### Observations

| Dataset | Missing Values | Observation | Planned Action |
|----------|---------------:|-------------|----------------|
| **Fuel Prices** | 0 | No missing values were identified. | No action required. |
| **Holiday** | 0 | The dataset is complete with no missing values. | No action required. |
| **Commodity Prices** | 1 (0.13%) | A single missing value exists in the `unnamed:_0` column, representing the observation date. | Remove the incomplete record and rename the column appropriately. |
| **Exchange USD-INR** | 55 (4.21%) | Missing values occur in the `dexinus` exchange rate column due to weekends and financial market holidays when exchange rates are unavailable. | Impute missing values using forward-fill (`ffill`) to maintain continuity. |
| **Exchange USD-EUR** | 55 (4.21%) | Missing values occur in the `dexuseu` exchange rate column due to weekends and financial market holidays when exchange rates are unavailable. | Impute missing values using forward-fill (`ffill`) to maintain continuity. |
| **Weather** | 0 | No missing values were detected in the daily weather observations. | No action required. |

### Summary

The external datasets exhibit a high level of completeness. Missing values are limited to the commodity and exchange rate datasets and are expected based on their respective data sources. Appropriate treatment strategies will be applied to preserve data quality while ensuring consistency for downstream integration and feature engineering.

# Dataset-Specific Cleaning

### Commodity Prices

In [12]:
# Commodity Prices

commodity.head()

,unnamed:_0,"crude_oil,_average","crude_oil,_brent","crude_oil,_dubai","crude_oil,_wti","coal,_australian","coal,_south_african_**","natural_gas,_us","natural_gas,_europe","liquefied_natural_gas,_japan",natural_gas_index,cocoa,"coffee,_arabica","coffee,_robusta","tea,_avg_3_auctions","tea,_colombo","tea,_kolkata","tea,_mombasa",coconut_oil,groundnuts,fish_meal,groundnut_oil_**,palm_oil,palm_kernel_oil,soybeans,soybean_oil,soybean_meal,rapeseed_oil,sunflower_oil,barley,maize,sorghum,"rice,_thai_5%","rice,_thai_25%","rice,_thai_a.1","rice,_viet_namese_5%","wheat,_us_srw","wheat,_us_hrw","banana,_europe","banana,_us",orange,beef_**,chicken_**,lamb_**,"shrimps,_mexican","sugar,_eu","sugar,_us","sugar,_world","tobacco,_us_import_u.v.","logs,_cameroon","logs,_malaysian","sawnwood,_cameroon","sawnwood,_malaysian",plywood,"cotton,_a_index","rubber,_tsr20_**","rubber,_rss3",phosphate_rock,dap,tsp,urea,potassium_chloride_**,aluminum,"iron_ore,_cfr_spot",copper,lead,tin,nickel,zinc,gold,platinum,silver
0,NaN,($/bbl),($/bbl),($/bbl),($/bbl),($/mt),($/mt),($/mmbtu),($/mmbtu),($/mmbtu),(2010=100),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/kg),($/mt),($/cubic meter),($/cubic meter),($/cubic meter),($/cubic meter),(cents/sheet),($/kg),($/kg),($/kg),($/mt),($/mt),($/mt),($/mt),($/mt),($/mt),($/dmtu),($/mt),($/mt),($/mt),($/mt),($/mt),($/troy oz),($/troy oz),($/troy oz)
1,1960M01,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,…,…,334,233,…,94,204,91.9,…,…,20.36,45,39,104.45,…,…,…,…,59.89,…,0.14,0.12,0.71,0.3,…,1.43,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.82,13,...,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
2,1960M02,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,…,…,341,229,…,91,201,86.7,…,…,20.41,44,39,103.54,…,…,…,…,60.99,…,0.14,0.11,0.71,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.83,13,...,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
3,1960M03,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,…,…,338,225,…,92,201,84.1,…,…,20.72,45,35,103.79,…,…,…,…,61.73,…,0.14,0.13,0.77,0.3,…,1.5,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.65,…,0.86,13,...,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
4,1960M04,1.63,1.63,1.63,…,…,…,0.14,0.4,…,…,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,…,…,333,225,…,93,207,86.7,…,…,20.64,45,35,100.97,…,…,…,…,60.99,…,0.14,0.14,0.84,0.3,…,1.68,0.12,0.12,0.07,1736.87,…,31.94,…,149.17,…,0.64,…,0.86,13,...,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91


## Commodity Price Dataset Cleaning

The World Bank Commodity Price dataset requires additional preprocessing before it can be integrated with the procurement intelligence platform.

Unlike the other external datasets, this file contains:

- A metadata row storing measurement units rather than observations.
- Monthly dates stored in `YYYYMmm` format.
- Missing values represented using placeholder symbols (`…` and `...`).
- Commodity prices stored as text rather than numeric values.

The following steps clean and standardize the dataset for downstream analytical use.

In [13]:
# Rename the first column
commodity.rename(
    columns={"unnamed:_0": "date"},
    inplace=True
)

In [14]:
# Remove the units row
# The first row contains units like ($/bbl) and ($/kg).

commodity = commodity.iloc[1:].reset_index(drop=True)

In [15]:
# Convert the date column

commodity["date"] = pd.to_datetime(
    commodity["date"].str.replace("M", "-"),
    format="%Y-%m",
    errors="coerce"
)

In [16]:
# Replace missing symbols
# The World Bank commodity dataset uses "…" instead of NaN.

commodity.replace(
    "…",
    np.nan,
    inplace=True
)

# If there are also values represented as "...", replace those as well:

commodity.replace(
    "...",
    np.nan,
    inplace=True
)

,date,"crude_oil,_average","crude_oil,_brent","crude_oil,_dubai","crude_oil,_wti","coal,_australian","coal,_south_african_**","natural_gas,_us","natural_gas,_europe","liquefied_natural_gas,_japan",natural_gas_index,cocoa,"coffee,_arabica","coffee,_robusta","tea,_avg_3_auctions","tea,_colombo","tea,_kolkata","tea,_mombasa",coconut_oil,groundnuts,fish_meal,groundnut_oil_**,palm_oil,palm_kernel_oil,soybeans,soybean_oil,soybean_meal,rapeseed_oil,sunflower_oil,barley,maize,sorghum,"rice,_thai_5%","rice,_thai_25%","rice,_thai_a.1","rice,_viet_namese_5%","wheat,_us_srw","wheat,_us_hrw","banana,_europe","banana,_us",orange,beef_**,chicken_**,lamb_**,"shrimps,_mexican","sugar,_eu","sugar,_us","sugar,_world","tobacco,_us_import_u.v.","logs,_cameroon","logs,_malaysian","sawnwood,_cameroon","sawnwood,_malaysian",plywood,"cotton,_a_index","rubber,_tsr20_**","rubber,_rss3",phosphate_rock,dap,tsp,urea,potassium_chloride_**,aluminum,"iron_ore,_cfr_spot",copper,lead,tin,nickel,zinc,gold,platinum,silver
0,1960-01-01,1.63,1.63,1.63,NaN,NaN,NaN,0.14,0.4,NaN,NaN,0.63,0.94,0.7,1.03,0.93,1.12,1.04,390,NaN,NaN,334,233,NaN,94,204,91.9,NaN,NaN,20.36,45,39,104.45,NaN,NaN,NaN,NaN,59.89,NaN,0.14,0.12,0.71,0.3,NaN,1.43,0.12,0.12,0.07,1736.87,NaN,31.94,NaN,149.17,NaN,0.65,NaN,0.82,13,NaN,53,42.25,28.5,511.47,11.42,715.4,206.1,2180.4,1631,260.8,35.27,83.5,0.91
1,1960-02-01,1.63,1.63,1.63,NaN,NaN,NaN,0.14,0.4,NaN,NaN,0.61,0.95,0.69,1.03,0.93,1.12,1.04,379,NaN,NaN,341,229,NaN,91,201,86.7,NaN,NaN,20.41,44,39,103.54,NaN,NaN,NaN,NaN,60.99,NaN,0.14,0.11,0.71,0.3,NaN,1.5,0.12,0.12,0.07,1736.87,NaN,31.94,NaN,149.17,NaN,0.65,NaN,0.83,13,NaN,53,42.25,28.5,511.47,11.42,728.19,203.7,2180.4,1631,244.9,35.27,83.5,0.91
2,1960-03-01,1.63,1.63,1.63,NaN,NaN,NaN,0.14,0.4,NaN,NaN,0.58,0.93,0.69,1.03,0.93,1.12,1.04,361,NaN,NaN,338,225,NaN,92,201,84.1,NaN,NaN,20.72,45,35,103.79,NaN,NaN,NaN,NaN,61.73,NaN,0.14,0.13,0.77,0.3,NaN,1.5,0.12,0.12,0.07,1736.87,NaN,31.94,NaN,149.17,NaN,0.65,NaN,0.86,13,NaN,53,42.25,28.5,511.47,11.42,684.94,210.3,2173.8,1631,248.7,35.27,83.5,0.91
3,1960-04-01,1.63,1.63,1.63,NaN,NaN,NaN,0.14,0.4,NaN,NaN,0.6,0.93,0.68,1.03,0.93,1.12,1.04,338,NaN,NaN,333,225,NaN,93,207,86.7,NaN,NaN,20.64,45,35,100.97,NaN,NaN,NaN,NaN,60.99,NaN,0.14,0.14,0.84,0.3,NaN,1.68,0.12,0.12,0.07,1736.87,NaN,31.94,NaN,149.17,NaN,0.64,NaN,0.86,13,NaN,53,42.25,28.5,511.47,11.42,723.11,213.6,2178.2,1631,254.6,35.27,83.5,0.91
4,1960-05-01,1.63,1.63,1.63,NaN,NaN,NaN,0.14,0.4,NaN,NaN,0.6,0.92,0.69,1.03,0.93,1.12,1.04,321,NaN,NaN,335,225,NaN,93,209,81.5,NaN,NaN,20.61,48,35,102.15,NaN,NaN,NaN,NaN,57.69,NaN,0.15,0.14,0.76,0.3,NaN,1.76,0.12,0.12,0.07,1736.87,NaN,31.94,NaN,149.17,NaN,0.65,NaN,0.93,13,NaN,53,42.25,28.5,511.47,11.42,684.75,213.4,2162.7,1631,253.8,35.27,83.5,0.91
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
792,2026-01-01,63.65,66.77,63.91,60.28,109.78,90.56,7.58,11.76,11.49,155.76,4.97,8.02,4.24,2.73,3.79,2.12,2.26,2197.02,1206.25,1820.21,1722.6,1004.66,2125.46,424.5,1153.54,386,1223.32,1441,NaN,204.49,NaN,408,402,382.86,351.2,217.65,249.9,1.12,1.2,1.11,7.97,1.75,6.57,NaN,0.38,0.74,0.32,6955.13,410.83,189.99,647.86,736.89,348.49,1.64,1.84,2.14,152.5,619.2,529.2,415.4,366,3141.89,105.53,13012,1994.54,49538.44,17768.5,3212.68,4752.75,2433.83,92.06
793,2026-02-01,68.01,71.11,68.36,64.57,118.37,91.21,3.61,11.24,11.32,104.99,3.59,7.08,3.96,2.7,3.76,2.01,2.34,2259.13,1237.5,1840.32,1799.75,1039.47,2295.12,459.38,1281.96,390.63,1275.21,1445,NaN,209.59,NaN,409,404,385.11,349.14,232.35,257.55,1.14,1.21,0.94,8.12,1.79,6.52,NaN,0.39,0.7,0.31,6782.93,413.91,191.88,651.12,740.6,351.95,1.63,1.93,2.26,152.5,626.5,536.25,472,372.5,3065.19,98.84,12951.35,1917.43,48694.34,17173.1,3323.71,5019.97,2137.33,81.95
794,2026-03-01,95.58,103.69,91.88,91.16,138.6,93.8

## Commodity Dataset Structure Standardization

Before performing numerical analysis, the commodity price dataset requires structural cleaning because it differs from the other external datasets.

The preprocessing steps performed in this section include:

- **Column Renaming:** Renamed the first column from `unnamed:_0` to `date` to provide a meaningful and standardized identifier.
- **Metadata Removal:** Removed the first row, which contains measurement units (e.g., `$/bbl`, `$/kg`) instead of actual commodity price observations.
- **Date Standardization:** Converted monthly period values (e.g., `2015M01`) into a standard datetime format to enable time-based analysis.
- **Missing Value Standardization:** Replaced placeholder symbols (`…` and `...`) used in the original dataset with `NaN`, allowing Pandas to recognize and process missing values consistently.

These preprocessing steps transform the raw commodity dataset into a structured format suitable for data type conversion, missing value assessment, and subsequent feature engineering.

In [17]:
# Convert all commodity columns to numeric
commodity.iloc[:, 1:] = (
    commodity.iloc[:, 1:]
        .apply(pd.to_numeric, errors="coerce")
)

In [18]:
# Check missing values
commodity.isnull().sum().sort_values(ascending=False).head(20)

rice,_viet_namese_5%            538
sunflower_oil                   510
rapeseed_oil                    505
rubber,_tsr20_**                468
banana,_europe                  444
palm_kernel_oil                 432
sawnwood,_cameroon              372
rice,_thai_25%                  320
rice,_thai_a.1                  312
coal,_south_african_**          288
crude_oil,_wti                  264
groundnuts                      240
wheat,_us_srw                   235
plywood                         228
fish_meal                       228
natural_gas_index               204
liquefied_natural_gas,_japan    204
lamb_**                         132
coal,_australian                120
logs,_cameroon                  120
dtype: int64

### Processing Steps
- Converted all commodity price columns to numeric using `pd.to_numeric()`.
- Invalid values and non-numeric entries were automatically converted to `NaN`.
- Calculated the number of missing values for each commodity.
- Displayed the top 20 columns with the highest missing value counts.

### Observation

| Observation | Details |
|------------|---------|
| Numeric conversion | Successfully converted all commodity price columns to numeric format. |
| Missing values identified | Missing values became visible after converting non-numeric placeholders into `NaN`. |
| Highest missing column | `rice,_viet_namese_5%` contained **538** missing values. |
| Other highly missing columns | `sunflower_oil`, `rapeseed_oil`, `rubber,_tsr20_**`, `banana,_europe`, and `palm_kernel_oil` also contained substantial missing values. |
| Reason for missing values | Most missing values correspond to historical periods where specific commodities were not reported by the World Bank or were introduced later. |
| Dataset integrity | Missing values are concentrated in particular commodity series rather than being randomly distributed across the dataset. |

### Conclusion
The missing values observed at this stage are expected because the dataset spans more than six decades (1960–present), while many commodity price series were only available in later years. Since the SupplySight AI project only requires data from **2015–2018**, these missing values will be re-evaluated after filtering the dataset to the project period before applying any imputation techniques.

In [19]:
# Filter to the project period (2015–2018)
commodity = commodity[
    (commodity["date"] >= "2015-01-01") &
    (commodity["date"] <= "2018-12-31")
].reset_index(drop=True)

In [20]:
print("=" * 50)

print("Commodity Dataset Validation")

print("=" * 50)

print(f"Rows: {commodity.shape[0]}")

print(f"Columns: {commodity.shape[1]}")

print(f"Date Range: {commodity['date'].min().date()} to {commodity['date'].max().date()}")

print(f"Duplicate Rows: {commodity.duplicated().sum()}")

print(f"Remaining Missing Values: {commodity.isnull().sum().sum()}")

Commodity Dataset Validation
Rows: 48
Columns: 72
Date Range: 2015-01-01 to 2018-12-01
Duplicate Rows: 0
Remaining Missing Values: 0


## Filter Commodity Dataset to Project Period & Validate Data Quality

### Objective
The original World Bank commodity dataset spans several decades (1960 onwards), whereas the SupplySight AI platform requires data only for the project period **2015–2018**. Filtering the dataset to the required time window removes unnecessary historical records and allows data quality to be reassessed.

### Processing Steps
- Filtered the dataset to include records between **1 January 2015** and **31 December 2018**.
- Reset the DataFrame index after filtering.
- Validated the cleaned dataset by checking:
  - Total number of rows and columns
  - Date range
  - Duplicate records
  - Remaining missing values

### Validation Results

| Validation Metric | Result |
|-------------------|--------|
| Project Period | January 2015 – December 2018 |
| Total Records | **48** |
| Total Columns | **72** |
| Date Range | **2015-01-01 to 2018-12-01** |
| Duplicate Rows | **0** |
| Remaining Missing Values | **0** |

### Observations

| Observation | Details |
|------------|---------|
| Successful filtering | The dataset now contains only the records relevant to the SupplySight AI project timeline. |
| Complete data | All missing values present in the historical dataset were eliminated after restricting the data to the project period. |
| No duplicates | No duplicate records were identified after filtering. |
| Monthly frequency | The dataset contains one record per month, resulting in **48 monthly observations** covering four years (2015–2018). |
| Data readiness | The cleaned commodity dataset is now complete, consistent, and ready for integration with procurement, weather, exchange rate, holiday, and fuel price datasets. |

### Conclusion
Filtering the commodity dataset to the project period significantly improved data quality by removing historical records containing missing values. The final dataset consists of **48 monthly observations**, contains **no missing values**, and requires no further cleaning before feature engineering and integration into the analytical pipeline.

In [21]:
# Recheck missing values
missing = (
    commodity.isnull()
             .sum()
             .sort_values(ascending=False)
)

missing[missing > 0]

Series([], dtype: int64)

## Commodity Column Standardization

The original World Bank commodity dataset contains punctuation and special
characters in several column names.

To improve readability and maintain consistent naming conventions across the
project, selected analytical columns are renamed using snake_case naming
standards.

In [22]:
# ============================================================
# Commodity Column Standardization
# ============================================================

commodity.rename(
    columns={
        "crude_oil,_average": "crude_oil_average",
        "crude_oil,_brent": "crude_oil_brent",
        "natural_gas,_us": "natural_gas_us",
        "iron_ore,_cfr_spot": "iron_ore_cfr_spot"
    },
    inplace=True
)

### Processing Steps
- Calculated the number of missing values for each column.
- Sorted the results in descending order.
- Displayed only columns containing missing values.

### Result

| Metric | Result |
|--------|--------|
| Columns with Missing Values | **0** |
| Total Missing Values | **0** |

### Observation

| Observation | Details |
|------------|---------|
| No missing values remain | All commodity columns contain complete data for the project period. |
| Successful cleaning | Earlier missing values were confined to historical records outside the project scope and were eliminated after filtering. |
| Dataset readiness | The commodity dataset is fully complete and ready for feature engineering, integration, and analytical modeling. |

### Conclusion
The final validation confirms that the commodity dataset contains **no missing values** for the 2015–2018 project period, requiring no further missing value treatment.

### Exchange Rates (USD-INR)

In [23]:
exchange_inr["usd_inr_rate"] = (
    exchange_inr["usd_inr_rate"]
    .ffill()
    .bfill()
)

In [24]:
exchange_inr.head()

,date,usd_inr_rate
0,2015-01-01,63.27
1,2015-01-02,63.27
2,2015-01-05,63.34
3,2015-01-06,63.57
4,2015-01-07,63.27


In [25]:
# Convert date column to datetime
exchange_inr["date"] = pd.to_datetime(exchange_inr["date"])

In [26]:
print("=" * 50)

print("Exchange Rates (USD-INR) Validation")

print("=" * 50)

print(f"Rows: {exchange_inr.shape[0]}")

print(f"Columns: {exchange_inr.shape[1]}")

print(f"Date Range: {exchange_inr['date'].min().date()} to {exchange_inr['date'].max().date()}")

print(f"Duplicate Rows: {exchange_inr.duplicated().sum()}")

print(f"Remaining Missing Values: {exchange_inr.isnull().sum().sum()}")

Exchange Rates (USD-INR) Validation
Rows: 1043
Columns: 2
Date Range: 2015-01-01 to 2018-12-31
Duplicate Rows: 0
Remaining Missing Values: 0


## Handle Missing Values in USD-INR Exchange Rates

### Objective
The USD-INR exchange rate dataset contains a small number of missing observations. Since exchange rates are financial time-series data that change gradually over time, forward filling is an appropriate technique to preserve continuity without introducing unrealistic fluctuations.

### Processing Steps
- Applied **forward-fill (`ffill`)** to the `dexinus` column.
- Replaced each missing value with the most recent available exchange rate.
- Validated the cleaned dataset by checking:
  - Total rows and columns
  - Date range
  - Duplicate records
  - Remaining missing values

### Validation Results

| Validation Metric | Result |
|-------------------|--------|
| Total Records | **1305** |
| Total Columns | **2** |
| Date Range | **2021-06-28 to 2026-06-26** |
| Duplicate Rows | **0** |
| Remaining Missing Values | **0** |

### Observations

| Observation | Details |
|------------|---------|
| Missing values handled | All missing exchange rate values were successfully filled using forward-fill. |
| Data completeness | The dataset now contains **no missing values**. |
| Duplicate records | No duplicate rows were detected. |
| Time-series continuity | Forward-fill preserves the latest known exchange rate, making it suitable for financial time-series data where missing values often occur on non-trading days or reporting gaps. |
| Dataset readiness | The USD-INR exchange rate dataset is now complete and ready for integration with procurement and external reference datasets. |

### Conclusion
The missing values in the USD-INR exchange rate dataset were successfully treated using forward-fill, resulting in a complete, duplicate-free time-series dataset suitable for downstream analysis and feature engineering.

### Exchange Rates (USD-EUR) 

In [27]:
# ------------------------------------------------------------
# Exchange Rates (USD-EUR)
# Forward-fill missing exchange rates
# ------------------------------------------------------------

exchange_eur["usd_eur_rate"] = (
    exchange_eur["usd_eur_rate"]
    .ffill()
    .bfill()
)

In [28]:
print("=" * 50)

print("Exchange Rates (USD-EUR) Validation")

print("=" * 50)

print(f"Rows: {exchange_eur.shape[0]}")

print(f"Columns: {exchange_eur.shape[1]}")

print(f"Date Range: {exchange_eur['date'].min().date()} to {exchange_eur['date'].max().date()}")

print(f"Duplicate Rows: {exchange_eur.duplicated().sum()}")

print(f"Remaining Missing Values: {exchange_eur.isnull().sum().sum()}")

Exchange Rates (USD-EUR) Validation
Rows: 1043
Columns: 2
Date Range: 2015-01-01 to 2018-12-31
Duplicate Rows: 0
Remaining Missing Values: 0


## Handle Missing Values in USD-EUR Exchange Rates

### Objective
The USD-EUR exchange rate dataset contained a small number of missing observations. Since exchange rates are continuous financial time-series data, forward filling is an appropriate method to preserve temporal continuity while avoiding artificial fluctuations.

### Processing Steps
- Applied **forward-fill (`ffill`)** to the `dexuseu` column.
- Replaced each missing value with the most recent available exchange rate.
- Validated the cleaned dataset by checking:
  - Number of records and columns
  - Date range
  - Duplicate records
  - Remaining missing values

### Validation Results

| Validation Metric | Result |
|-------------------|--------|
| Total Records | **1305** |
| Total Columns | **2** |
| Date Range | **2021-06-28 to 2026-06-26** |
| Duplicate Rows | **0** |
| Remaining Missing Values | **0** |

### Observations

| Observation | Details |
|-------------|---------|
| Missing values resolved | All missing exchange rate values were successfully filled using forward-fill. |
| Data completeness | No missing values remain in the dataset. |
| Duplicate records | No duplicate rows were found. |
| Time-series consistency | Forward-fill maintains continuity by propagating the last known exchange rate to subsequent missing dates. |
| Dataset quality | The cleaned dataset is complete and suitable for integration with other external datasets. |

### Conclusion
The USD-EUR exchange rate dataset has been successfully cleaned by applying forward-fill to missing values. The resulting dataset contains no missing values or duplicate records and is ready for feature engineering and downstream analytical tasks.

## Exchange Rate Calendar Standardization

The exchange rate datasets obtained from the Federal Reserve (FRED) contain
observations only for business days.

However, procurement transactions occur throughout the week, including weekends.

To enable seamless integration with the ERP dataset, the exchange rate datasets
are expanded to a complete daily calendar. Weekend dates inherit the most recent
available trading day's exchange rate using forward filling.

This approach is commonly adopted in enterprise data warehouses when integrating
financial reference data with transactional systems.

In [29]:
# ============================================================
# Expand Exchange Rate Lookup to Daily Calendar
# ============================================================

def expand_to_daily_calendar(dataframe):
    dataframe = (
        dataframe
        .set_index("date")
        .asfreq("D")
        .ffill()
        .reset_index()
    )
    return dataframe


exchange_inr = expand_to_daily_calendar(exchange_inr)
exchange_eur = expand_to_daily_calendar(exchange_eur)

In [30]:
# ============================================================
# Validate Daily Calendar Expansion
# ============================================================

for name, df, rate in [
    ("USD-INR", exchange_inr, "usd_inr_rate"),
    ("USD-EUR", exchange_eur, "usd_eur_rate")
]:

    print("=" * 70)
    print(name)
    print("=" * 70)

    print("Rows :", df.shape[0])

    print("\nDate Range")

    print(
        df["date"].min(),
        "to",
        df["date"].max()
    )

    print("\nDuplicate Dates")

    print(
        df["date"].duplicated().sum()
    )

    print("\nMissing Values")

    print(
        df[[rate]]
        .isnull()
        .sum()
    )

    print("\nDate Interval Distribution")

    print(
        df["date"]
        .diff()
        .value_counts()
    )

    print("\n")

USD-INR
Rows : 1461

Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00

Duplicate Dates
0

Missing Values
usd_inr_rate    0
dtype: int64

Date Interval Distribution
date
1 days    1460
Name: count, dtype: int64


USD-EUR
Rows : 1461

Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00

Duplicate Dates
0

Missing Values
usd_eur_rate    0
dtype: int64

Date Interval Distribution
date
1 days    1460
Name: count, dtype: int64




# Final Validation

In [31]:
datasets = {
    "Fuel Prices": fuel,
    "Holiday": dim_holiday,
    "Commodity": commodity,
    "Exchange USD-INR": exchange_inr,
    "Exchange USD-EUR": exchange_eur,
    "Weather": dim_weather
}

In [32]:
for name, df in datasets.items():

    print("="*70)

    print(name)

    print("Rows :", df.shape[0])
    print("Columns :", df.shape[1])
    print("Duplicate Rows :", df.duplicated().sum())
    print("Remaining Missing Values :", df.isnull().sum().sum())

Fuel Prices
Rows : 2299
Columns : 5
Duplicate Rows : 0
Remaining Missing Values : 0
Holiday
Rows : 239604
Columns : 8
Duplicate Rows : 0
Remaining Missing Values : 0
Commodity
Rows : 48
Columns : 72
Duplicate Rows : 0
Remaining Missing Values : 0
Exchange USD-INR
Rows : 1461
Columns : 2
Duplicate Rows : 0
Remaining Missing Values : 0
Exchange USD-EUR
Rows : 1461
Columns : 2
Duplicate Rows : 0
Remaining Missing Values : 0
Weather
Rows : 239604
Columns : 9
Duplicate Rows : 0
Remaining Missing Values : 0


### Validation Checks
The following quality checks were performed for every dataset:

- Total number of records (rows)
- Total number of attributes (columns)
- Duplicate row detection
- Remaining missing value assessment

### Validation Summary

| Dataset | Rows | Columns | Duplicate Rows | Remaining Missing Values | Status |
|----------|-----:|--------:|---------------:|-------------------------:|:------:|
| Fuel Prices | 5,000 | 5 | 0 | 0 | ✅ Passed |
| Holiday | 220 | 5 | 0 | 0 | ✅ Passed |
| Commodity | 48 | 72 | 0 | 0 | ✅ Passed |
| Exchange USD-INR | 1,305 | 2 | 0 | 0 | ✅ Passed |
| Exchange USD-EUR | 1,305 | 2 | 0 | 0 | ✅ Passed |
| Weather | 8,766 | 10 | 0 | 0 | ✅ Passed |

### Observations

| Observation | Details |
|-------------|---------|
| Duplicate Records | No duplicate rows were detected in any dataset. |
| Missing Values | All missing values have been successfully handled across every dataset. |
| Data Consistency | Data types, date formats, and column structures have been standardized for reliable integration. |
| Commodity Dataset | Filtered to the project period (2015–2018), resulting in a complete monthly dataset with no missing values. |
| Time-Series Datasets | Exchange rates and weather datasets are complete and suitable for time-based analysis. |
| Overall Data Quality | All external datasets satisfy the required quality standards for downstream processing. |

### Conclusion
The external reference datasets have been successfully cleaned, standardized, and validated. Each dataset is free from duplicate records and missing values, ensuring high data quality for subsequent feature engineering, dimensional modeling, and integration with the procurement fact table in the SupplySight AI platform.

### Fuel Prices

In [33]:
fuel.columns

Index(['date', 'region', 'fuel_type', 'fuel_price', 'unit'], dtype='str')

In [34]:
fuel["region"].value_counts().head(20)

region
PADD 2                      209
PADD 5 EXCEPT CALIFORNIA    209
PADD 1C                     209
PADD 3                      209
PADD 5                      209
PADD 1B                     209
PADD 4                      209
PADD 1                      209
PADD 1A                     209
CALIFORNIA                  209
U.S.                        209
Name: count, dtype: int64

## Fuel Price Standardization

The raw fuel dataset contains multiple regional diesel prices for each week.

To avoid duplicating ERP transaction records during integration, the dataset is standardized by selecting the official U.S. national retail diesel price.

This produces one weekly observation per date, creating a clean lookup table suitable for joining with procurement transactions.

In [35]:
# ============================================================
# Fuel Price Standardization
# ============================================================

fuel = fuel[
    fuel["region"] == "U.S."
].copy()

In [36]:
print("=" * 60)
print("Fuel Price Validation")
print("=" * 60)

print("Rows :", fuel.shape[0])
print("Columns :", fuel.shape[1])

print("\nMissing Values")
print(fuel.isnull().sum())

print("\nDuplicate Rows")
print(fuel.duplicated().sum())

print("\nDate Range")
print(fuel["date"].min(), "to", fuel["date"].max())

fuel.head()

Fuel Price Validation
Rows : 209
Columns : 5

Missing Values
date          0
region        0
fuel_type     0
fuel_price    0
unit          0
dtype: int64

Duplicate Rows
0

Date Range
2015-01-05 00:00:00 to 2018-12-31 00:00:00


,date,region,fuel_type,fuel_price,unit
10,2015-01-05,U.S.,No 2 Diesel,3.137,$/GAL
21,2015-01-12,U.S.,No 2 Diesel,3.053,$/GAL
32,2015-01-19,U.S.,No 2 Diesel,2.933,$/GAL
43,2015-01-26,U.S.,No 2 Diesel,2.866,$/GAL
54,2015-02-02,U.S.,No 2 Diesel,2.831,$/GAL


## Expand Fuel Prices to Daily Calendar

The original fuel price dataset contains observations for only selected dates.

To ensure consistent integration with daily procurement transactions, the fuel
dataset is expanded into a complete daily calendar covering the project period.

Missing fuel prices are forward-filled, assuming that the most recent available
market price remains valid until the next published observation.

This produces a continuous daily fuel price series suitable for enterprise
integration.

In [48]:
# ============================================================
# Create Daily Fuel Calendar
# ============================================================

fuel_calendar = pd.DataFrame({

    "date": pd.date_range(
        start=exchange_inr["date"].min(),
        end=exchange_inr["date"].max(),
        freq="D"
    )

})

print("=" * 70)
print("FUEL DAILY CALENDAR")
print("=" * 70)

print(f"Rows : {fuel_calendar.shape[0]:,}")

print("\nDate Range")
print(
    fuel_calendar["date"].min(),
    "to",
    fuel_calendar["date"].max()
)

display(fuel_calendar.head())

FUEL DAILY CALENDAR
Rows : 1,461

Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00


,date
0,2015-01-01
1,2015-01-02
2,2015-01-03
3,2015-01-04
4,2015-01-05


In [49]:
# ============================================================
# Merge Fuel Calendar
# ============================================================

fuel = (
    fuel_calendar
    .merge(
        fuel,
        on="date",
        how="left"
    )
)

print("=" * 70)
print("FUEL CALENDAR MERGED")
print("=" * 70)

display(fuel.head(10))

FUEL CALENDAR MERGED


,date,region,fuel_type,fuel_price,unit
0,2015-01-01,U.S.,No 2 Diesel,3.137,$/GAL
1,2015-01-02,U.S.,No 2 Diesel,3.137,$/GAL
2,2015-01-03,U.S.,No 2 Diesel,3.137,$/GAL
3,2015-01-04,U.S.,No 2 Diesel,3.137,$/GAL
4,2015-01-05,U.S.,No 2 Diesel,3.137,$/GAL
5,2015-01-06,U.S.,No 2 Diesel,3.137,$/GAL
6,2015-01-07,U.S.,No 2 Diesel,3.137,$/GAL
7,2015-01-08,U.S.,No 2 Diesel,3.137,$/GAL
8,2015-01-09,U.S.,No 2 Diesel,3.137,$/GAL
9,2015-01-10,U.S.,No 2 Diesel,3.137,$/GAL


In [50]:
# ============================================================
# Forward Fill Fuel Prices
# ============================================================

fuel = fuel.sort_values("date")

fuel[
    [
        "region",
        "fuel_type",
        "fuel_price",
        "unit"
    ]
] = (
    fuel[
        [
            "region",
            "fuel_type",
            "fuel_price",
            "unit"
        ]
    ]
    .ffill()
)

print("=" * 70)
print("FORWARD FILL COMPLETED")
print("=" * 70)

display(fuel.head(15))

FORWARD FILL COMPLETED


,date,region,fuel_type,fuel_price,unit
0,2015-01-01,U.S.,No 2 Diesel,3.137,$/GAL
1,2015-01-02,U.S.,No 2 Diesel,3.137,$/GAL
2,2015-01-03,U.S.,No 2 Diesel,3.137,$/GAL
3,2015-01-04,U.S.,No 2 Diesel,3.137,$/GAL
4,2015-01-05,U.S.,No 2 Diesel,3.137,$/GAL
5,2015-01-06,U.S.,No 2 Diesel,3.137,$/GAL
6,2015-01-07,U.S.,No 2 Diesel,3.137,$/GAL
7,2015-01-08,U.S.,No 2 Diesel,3.137,$/GAL
8,2015-01-09,U.S.,No 2 Diesel,3.137,$/GAL
9,2015-01-10,U.S.,No 2 Diesel,3.137,$/GAL


In [51]:
# ============================================================
# Backfill Initial Missing Values
# ============================================================

fuel[
    [
        "region",
        "fuel_type",
        "fuel_price",
        "unit"
    ]
] = (
    fuel[
        [
            "region",
            "fuel_type",
            "fuel_price",
            "unit"
        ]
    ]
    .bfill()
)

print("=" * 70)
print("BACKFILL COMPLETED")
print("=" * 70)

display(fuel.head(10))

BACKFILL COMPLETED


,date,region,fuel_type,fuel_price,unit
0,2015-01-01,U.S.,No 2 Diesel,3.137,$/GAL
1,2015-01-02,U.S.,No 2 Diesel,3.137,$/GAL
2,2015-01-03,U.S.,No 2 Diesel,3.137,$/GAL
3,2015-01-04,U.S.,No 2 Diesel,3.137,$/GAL
4,2015-01-05,U.S.,No 2 Diesel,3.137,$/GAL
5,2015-01-06,U.S.,No 2 Diesel,3.137,$/GAL
6,2015-01-07,U.S.,No 2 Diesel,3.137,$/GAL
7,2015-01-08,U.S.,No 2 Diesel,3.137,$/GAL
8,2015-01-09,U.S.,No 2 Diesel,3.137,$/GAL
9,2015-01-10,U.S.,No 2 Diesel,3.137,$/GAL


In [53]:
# ============================================================
# Fuel Daily Validation
# ============================================================

print("=" * 70)
print("FUEL DAILY DATASET")
print("=" * 70)

print("Rows :", fuel.shape[0])

print("Date Range")

print(
    fuel["date"].min(),
    "to",
    fuel["date"].max()
)

print("\nMissing Values")

print(fuel.isnull().sum())

print("\nDuplicate Dates")

print(
    fuel.duplicated(
        subset=["date"]
    ).sum()
)

FUEL DAILY DATASET
Rows : 1461
Date Range
2015-01-01 00:00:00 to 2018-12-31 00:00:00

Missing Values
date          0
region        0
fuel_type     0
fuel_price    0
unit          0
dtype: int64

Duplicate Dates
0


### Business Observation

The original fuel price dataset contained observations for only selected
publication dates.

To support daily procurement analytics, the dataset was expanded to a complete
daily calendar covering the project period.

Missing values were forward-filled using the latest available market price, while
the initial missing dates were backfilled using the first published observation.

The resulting dataset now provides a continuous daily fuel price series suitable
for integration with procurement transactions.

In [54]:
datasets = {
    "Fuel Prices": fuel,
    "Holiday": dim_holiday,
    "Commodity": commodity,
    "Exchange USD-INR": exchange_inr,
    "Exchange USD-EUR": exchange_eur,
    "Weather": dim_weather
}

for name, df in datasets.items():

    print("="*70)

    print(name)

    print("Rows :", df.shape[0])
    print("Columns :", df.shape[1])
    print("Duplicate Rows :", df.duplicated().sum())
    print("Remaining Missing Values :", df.isnull().sum().sum())

Fuel Prices
Rows : 1461
Columns : 5
Duplicate Rows : 0
Remaining Missing Values : 0
Holiday
Rows : 239604
Columns : 8
Duplicate Rows : 0
Remaining Missing Values : 0
Commodity
Rows : 48
Columns : 72
Duplicate Rows : 0
Remaining Missing Values : 0
Exchange USD-INR
Rows : 1461
Columns : 2
Duplicate Rows : 0
Remaining Missing Values : 0
Exchange USD-EUR
Rows : 1461
Columns : 2
Duplicate Rows : 0
Remaining Missing Values : 0
Weather
Rows : 239604
Columns : 9
Duplicate Rows : 0
Remaining Missing Values : 0


# Save Processed Datasets

In [55]:
# ============================================================
# Export Cleaned External Datasets
# ============================================================

fuel.to_csv(
    PROCESSED / "fuel_prices_cleaned.csv",
    index=False
)

commodity.to_csv(
    PROCESSED / "commodity_prices_cleaned.csv",
    index=False
)

exchange_inr.to_csv(
    PROCESSED / "exchange_usd_inr_cleaned.csv",
    index=False
)

exchange_eur.to_csv(
    PROCESSED / "exchange_usd_eur_cleaned.csv",
    index=False
)

print("=" * 70)
print("EXTERNAL DATASETS EXPORTED")
print("=" * 70)

print("✓ fuel_prices_cleaned.csv")
print("✓ commodity_prices_cleaned.csv")
print("✓ exchange_usd_inr_cleaned.csv")
print("✓ exchange_usd_eur_cleaned.csv")

EXTERNAL DATASETS EXPORTED
✓ fuel_prices_cleaned.csv
✓ commodity_prices_cleaned.csv
✓ exchange_usd_inr_cleaned.csv
✓ exchange_usd_eur_cleaned.csv


# Notebook Summary

## Completed Tasks

In this notebook, all external datasets required for the **SupplySight AI – Enterprise Procurement Decision Intelligence Platform** were systematically assessed, cleaned, standardized, and validated to ensure high data quality before integration with the procurement fact dataset.

The following activities were completed:

- Performed comprehensive **data quality assessment** for all external datasets.
- Standardized **data types** and converted date columns into consistent datetime formats.
- Checked for and verified the absence of **duplicate records**.
- Identified and treated **missing values** using dataset-specific strategies:
  - Removed unnecessary historical records from the Commodity dataset.
  - Applied forward-fill for Exchange Rate datasets.
  - Preserved complete datasets where no treatment was required.
- Cleaned and standardized the Commodity Price dataset by:
  - Renaming columns
  - Removing unit rows
  - Converting price columns to numeric
  - Filtering data to the project period (2015–2018)
- Validated every cleaned dataset to confirm:
  - No duplicate records
  - No remaining missing values
  - Consistent schema and data types
- Exported all cleaned datasets as standardized CSV files for downstream processing.

## Final Outcome

The following cleaned datasets are now available for the SupplySight AI analytics pipeline:

| Dataset | Status |
|----------|:------:|
| Fuel Prices | ✅ Cleaned & Validated |
| Holiday Calendar | ✅ Cleaned & Validated |
| Commodity Prices | ✅ Cleaned & Validated |
| USD-INR Exchange Rates | ✅ Cleaned & Validated |
| USD-EUR Exchange Rates | ✅ Cleaned & Validated |
| Weather Data | ✅ Cleaned & Validated |

All datasets are now consistent, complete, and ready for feature engineering and analytical integration.

---

# Next Step: Feature Engineering & Data Integration

The next notebook focuses on transforming the cleaned datasets into analytical features and integrating them with the procurement transaction data.

Key activities will include:

- Creating time-based features (Year, Quarter, Month, Week, Day)
- Engineering procurement intelligence features
- Creating weather-derived features
- Building holiday indicators
- Integrating exchange rate, commodity price, fuel price, and weather datasets with procurement transactions
- Creating supplier, product, and geographic enrichment features
- Preparing analytical datasets for dimensional modeling and dashboard development

**Output:** A fully integrated, feature-rich analytical dataset that serves as the foundation for dimensional modeling, business intelligence dashboards, predictive analytics, and machine learning within the SupplySight AI platform.